# 03 - Extra models and AMBER (Google Colab, GPU)

Extracts the same features as notebook 01 (last prompt token, every layer, yes/no scores) for other models and for AMBER. Everything calls `src/extract_features_multi.py`. Change `MODEL_KEY` / `DATASET` in the config and re-run from the top.

Runs used: `qwen2vl-2b` + `amber`; `smolvlm-500m` + `pope` and `amber`; `qwen25vl-3b` + `pope` / `amber`; optionally `llava15-7b` (4-bit).

Run the smoke test cell first: it prints the yes/no token ids, `answer_mass` (should be close to 1) and a time estimate. Finished 100-question shards are skipped on restart.

In [ ]:
# Config
MODEL_KEY = "smolvlm-500m"      # qwen2vl-2b | qwen25vl-3b | smolvlm-500m | llava15-7b
DATASET = "pope"                # "pope" | "amber"
POPE_SUBSET_SIZE = 3000         # first N rows of the seeded order (None = all 9000)
AMBER_PER_SUBSET = 1000         # random queries per AMBER subset
LAYERS = "all"                  # "all" | "every:4" | "0,8,16,-1"
REPO_URL = "https://github.com/RUANLASS/SC4001Assignment_2026.git"
BRANCH = "extra-models-amber"
USE_DRIVE = True                # write outputs to Google Drive
DRIVE_DIR = "/content/drive/MyDrive/sc4001_extra"
AMBER_HF_IMAGES = None          # fallback image source, e.g. "visual-preference/AMBER"


## a. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found")
if not Path("src/extract_features_multi.py").exists():
    target = Path("/content/SC4001Assignment_2026")
    if not target.exists():
        r = subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(target)], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert (target / "src" / "extract_features_multi.py").exists(), "clone failed (private repo? wrong BRANCH?): upload the repo folder instead"
    os.chdir(target)
print("working dir:", os.getcwd())

In [ ]:
!pip install -q -r requirements-extra.txt
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU, then re-run"
OUT_DIR = "results/cached_features"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive"); OUT_DIR = DRIVE_DIR; os.makedirs(OUT_DIR, exist_ok=True)
print("outputs ->", OUT_DIR)
!python src/extract_features_multi.py --list-models

## b. AMBER data (skip for POPE)
Metadata and ground truth come from the AMBER GitHub repo; images from the Google-Drive zip linked in its README. If `gdown` hits a quota, set `AMBER_HF_IMAGES = "visual-preference/AMBER"` in the config, or unzip the images by hand into `data/amber_images/`.

In [ ]:
if DATASET == "amber":
    if not Path("data/amber_repo").exists():
        !git clone -q https://github.com/junyangwang0410/AMBER data/amber_repo
    have_images = Path("data/amber_images").exists() and any(Path("data/amber_images").rglob("AMBER_*.jpg"))
    if AMBER_HF_IMAGES is None and not have_images:
        import gdown
        gdown.download(id="1MaCHgtupcZUjf007anNl4_MV0o4DjXvl", output="data/amber_images.zip", quiet=False)
        !unzip -q -o data/amber_images.zip -d data/amber_images
    AMBER_ARGS = f"--amber-hf-images {AMBER_HF_IMAGES}" if AMBER_HF_IMAGES else ""
    # Label distribution per subset
    !python src/extract_features_multi.py --dataset amber --inspect --per-subset {AMBER_PER_SUBSET} {AMBER_ARGS}
else:
    AMBER_ARGS = ""


## c. Smoke test (20 questions, writes nothing)
Check that the token ids look right, `mean answer_mass` is close to 1, and the time estimate is acceptable (otherwise lower `POPE_SUBSET_SIZE` / `AMBER_PER_SUBSET`).

In [ ]:
SIZE_ARGS = f"--subset-size {POPE_SUBSET_SIZE}" if (DATASET == "pope" and POPE_SUBSET_SIZE) else ""
COMMON = f"--model {MODEL_KEY} --dataset {DATASET} {SIZE_ARGS} --per-subset {AMBER_PER_SUBSET} --layers {LAYERS} --out-dir {OUT_DIR} {AMBER_ARGS}"
!python src/extract_features_multi.py {COMMON} --smoke 20

## d. Full run (resumable)

In [ ]:
!python src/extract_features_multi.py {COMMON}

## e. Output files
`{dataset}_{model}_features.pt` (not committed), `{dataset}_{model}_metadata.parquet` and `{dataset}_{model}_run_config.json`. Copy them into `results/cached_features/` and run the evaluation, e.g. `python src/probe_transfer_eval.py --models qwen2vl-2b smolvlm-500m --out-dir results/final/cross_model_logreg`.
The verification printout should show 0 missing, 0 duplicates, 0 non-finite values and no `!!! FLAG` line.

In [ ]:
!ls -la {OUT_DIR}